# Messages

In [49]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pathlib import Path

from langchain_core.messages import HumanMessage
from rich import print as rprint
import asyncio
import time
import base64

# 从.env文件中加载环境变量
load_dotenv(Path('../.env'), override=True)
DASHSCOPE_MODEL_NAME = "qwen3.7-plus"
DASHSCOPE_MODEL_PROVIDER = os.getenv("DASHSCOPE_MODEL_PROVIDER")
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model='qwen3.7-plus',
    model_provider='openai',
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # TODO 这里没有正确开启深度思考
    extra_body={
        "enable_thinking": True  # 启用思考模式
    }
)


def encode_image(img_path, img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode("utf-8")}"

## 传统content传入多模态数据
- 部分模型无法识别

In [21]:
img_base64 = encode_image('./image_test.png', img_type='png')

result_stream = model.stream([
    HumanMessage(content=[
        {'type': 'text', 'text': '这张图片里的是什么？'},
        {'type': 'image_url', 'image_url': img_base64},
    ]),
])

for resp in result_stream:
    print(resp.content, end='', flush=True)

这张图片里展示的是 **雅诗兰黛（Estée Lauder）的 Double Wear 持妆粉底液**（Double Wear Stay-in-Place Makeup）。

从图片特征可以看出：
1.  **瓶身设计**：经典的方形玻璃瓶身，里面装着米色/肤色的粉底液。
2.  **标签**：瓶身中间有金色的标签，上面隐约可以看到 "Estée Lauder" 和 "Double Wear" 的字样。
3.  **瓶盖**：透明的方形瓶盖，带有金色的泵头设计。

这款粉底液通常被称为“DW粉底液”，以其高遮瑕力和持久持妆效果而闻名。

## 老式content_blocks多模态输入
- 统一多模态调用输入。传统方式，将多模态数据和文本消息放入到content中，但有些模型(像ChatGPT, Claude)无法识别，而content_blocks是新版本统一方法

In [25]:
img_base64 = encode_image('./image_test.png', img_type='png')

result_stream = model.stream([
    HumanMessage(content_blocks=[
        {'type': 'text', 'text': '这张图片里的是什么？'},
        {'type': 'image', 'url': img_base64},
    ]),
])

for resp in result_stream:
    print(resp.content, end='', flush=True)

这张图片展示的是一瓶 **雅诗兰黛（Estée Lauder）的 Double Wear 持妆粉底液**（通常被大家简称为“DW粉底液”）。

从图片特征可以看出来：
1.  **瓶身设计**：它有着标志性的长方形透明玻璃瓶身。
2.  **标签**：瓶身中间有一个金色的标签，上面隐约可以看到 "Estée Lauder" 和 "Double Wear" 的字样。
3.  **内容物**：瓶内装着米色/肤色的液体，这是粉底液的颜色。
4.  **瓶盖**：顶部是透明带有金色内管的按压式泵头设计（或者是这种风格的盖子）。

这是一款非常著名的主打持久、遮瑕的粉底液产品。

## 新版content_blocks多模态输入

In [27]:
from langchain_core.messages.content import create_text_block, create_image_block

img_base64 = encode_image('./image_test.png', img_type='png')

result_stream = model.stream([
    HumanMessage(content_blocks=[
        create_text_block("这张图片是什么？"),
        # 这里编码后前缀是data:image，这是url，不是纯base64，所以关键字参数用url，不用base64
        create_image_block(url=img_base64, mime_type='image/png'),
    ]),
])

for resp in result_stream:
    print(resp.content, end='', flush=True)

这张图片展示的是 **雅诗兰黛（Estée Lauder）的 Double Wear 持妆粉底液**（Double Wear Stay-in-Place Makeup）。

从图片特征来看：
1.  **瓶身设计**：采用了标志性的方形透明玻璃瓶身，可以看到里面米色/肤色的粉底液液体。
2.  **标签**：瓶身中间有金色的标签，上面印有 "Estée Lauder" 品牌名和 "Double Wear" 产品名。
3.  **外观**：这是非常经典的一款粉底液，通常被称为“DW粉底液”，以持妆时间长、遮瑕力高而闻名。

图片中的光影效果（暖色调背景）是为了展示产品的质感和色号。

## content_blocks格式化输出
- 在深度思考时，不同模型的响应不同，比如`deepseek-v4-flash`返回的思考内容是在`response.additional_kwargs.resoning_content`，当我们使用不同厂商模型时，结构可能完全不一致，而`content_blocks`统一了输出结构:

In [48]:
img_base64 = encode_image('./image_test.png', img_type='png')

result_stream = model.stream(input=[
    HumanMessage(content_blocks=[
        create_text_block("为图片中的产品写一个创意广告语"),
        # 这里编码后前缀是data:image，这是url，不是纯base64，所以关键字参数用url，不用base64
        create_image_block(url=img_base64, mime_type='image/png'),
    ]),
])

print_content = {}

for resp in result_stream:
    if resp.content_blocks:
        for block in resp.content_blocks or ():
            if block['type'] == 'reasoning':
                print(f"[思考] {block.get('reasoning', '')}", end='', flush=True)
            elif block['type'] == 'text':
                print(block['text'], end='', flush=True)


这张图片展示的是雅诗兰黛（Estée Lauder）经典的 **Double Wear 持妆粉底液**。从图片的光影、金色的瓶盖以及自然的肤色来看，这款产品主打的是**高端质感、持久持妆和自然无瑕**。

以下为您提供几个不同角度的创意广告语：

**1. 强调“持久与时间”（核心卖点）：**
*   **“时光流转，妆颜如初。24小时持妆，无惧岁月漫长。”**
*   **“从晨光微露到暮色四合，你的美，无需补妆。”**
*   **“把时间定格在最美的一刻，Double Wear，持妆王者。”**

**2. 强调“质感与光影”（视觉美感）：**
*   **“金致瓶身，锁住原生光采。一抹无瑕，如丝般顺滑。”**
*   **“不仅是粉底，更是肌肤的金色外衣。焕现高级裸妆感。”**
*   **“捕捉每一缕光，雕琢每一寸肌。奢宠底妆，由此开始。”**

**3. 强调“自信与态度”（情感共鸣）：**
*   **“无惧镜头审视，无惧汗水考验。持妆力，就是你的底气。”**
*   **“做自己的主角，全天候高光时刻。”**
*   **“不脱妆的自信，是给你最好的礼物。”**

**4. 简短有力的 Slogan（适合海报）：**
*   **“持妆无瑕，金致绽放。”**
*   **“你的第二层肌肤，全天候守护。”**
*   **“美力持久，无可挑剔。”**

**建议搭配场景：**
如果您是在社交媒体（如小红书/朋友圈）发布，建议使用：
> “✨**晨光到暮色，妆容始终如一。** 雅诗兰黛DW粉底液，给你24小时的无瑕底气，无惧高清镜头，做最自信的自己！💛”

In [47]:
response = model.invoke("你好，一句话回答")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)

==================== -> response <- ====================
content='你好，请问有什么我可以帮您的？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 14, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 118, 'rejected_prediction_tokens': None, 'text_tokens': 131}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 14}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-7099fbf1-bfe1-9110-8015-a5a59be1bb3b', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f6f7c-cd98-78f1-897c-282be310f193-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 131, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 118}}
==================== -> response.content <- ==========